# Iris 

This example contains code snippets that demonstrate how concrete
implementations of machine learning models may be integratied into
the AIMM environment as plugins. File `aimm_plugins/plug1.py`
contains a simple wrapper around sklearn's SVC implementation and we're
going to use this to host a simple iris-recognition service.

In [1]:
from aimm.client import repl

aimm = repl.AIMM()
await aimm.connect('ws://127.0.0.1:9999')

Username:  user
Password:  ········


In [2]:
from pprint import pprint
pprint(aimm.state)

{'actions': {}, 'models': {}}


In [3]:
m = await aimm.create_instance('plugins.sklearn_wrapper.SVC')
m

aimm.client.repl.Model<plugins.sklearn_wrapper.SVC>(instance_id=1)

In [4]:
pprint(aimm.state)

{'actions': {1: {'meta': {'args': [],
                          'call': 'create_instance',
                          'kwargs': {},
                          'model_type': 'plugins.sklearn_wrapper.SVC'},
                 'status': 'running'}},
 'models': {}}


In [5]:
await m.fit(repl.DataAccessArg('iris_inputs'), repl.DataAccessArg('iris_outputs'))
await m.predict(repl.DataAccessArg('iris_inputs'))

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [10]:
pprint(aimm.state)

{'actions': {1: {'meta': {'args': [],
                          'call': 'create_instance',
                          'kwargs': {},
                          'model_type': 'plugins.sklearn_wrapper.SVC'},
                 'run': {'status': 'complete'},
                 'status': 'complete'},
             2: {'meta': {'args': ["DataAccessArg(name='iris_inputs', args=[], "
                                   'kwargs={})',
                                   "DataAccessArg(name='iris_outputs', "
                                   'args=[], kwargs={})'],
                          'call': 'fit',
                          'kwargs': {},
                          'model': 1},
                 'run': {'data_access': {'0': {'status': 'complete'},
                                         '1': {'status': 'complete'}},
                         'status': 'complete'},
                 'status': 'complete'},
             3: {'meta': {'meta': {'args': ["DataAccessArg(name='iris_inputs', "
                 

## Local plugin execution

All plugins may be executed separate from the AIMM server. The following
cells show how a basic workflow of a machine learning model, starting
from instantiation, fitting and practical usage - all done through the
plugins interface. On it's own, this is not particularly interesting -
after all, it would have easier to achieve the same without using the plugin
interface and using sklearn's models directly. Still, this shows how
AIMM server interprets and uses plugins when performing actions.

In [7]:
from aimm import plugins

In [8]:
plugins.initialize({'names': ['plugins.sklearn_wrapper']})
svc_type = 'plugins.sklearn_wrapper.SVC'
model = plugins.exec_instantiate(svc_type)

x = plugins.exec_data_access('iris_inputs')
y = plugins.exec_data_access('iris_outputs')

model = plugins.exec_fit(svc_type, model, lambda s: None, x, y)

In [11]:
index = 100

print('prediction:', plugins.exec_predict(svc_type, model, lambda s: None, x[index].reshape(1, -1))[1])
print('correct:', y[index])

prediction: [2]
correct: 2
